# 02. Stop Matching

**Scope of this notebook:** determine which scheduled stop a vehicle is at or approaching.

**Primary strategy:** trust the GTFS-Realtime feed's own reported `stop_id` / `current_stop_sequence` / `current_status` fields directly since MBTA already resolves this server-side for most trips.

**Fallback strategy:** the distance + monotonic-sequence inference approach from the original exploration is kept only for pings where the primary fields are null, only Section C below confirms the fallback can actually help those rows at all.


## A. Primary Strategy — Enriching Reported Stops

**Question:** For pings where MBTA already reports `stop_id`, can we simply join against `stops.txt` for a human-readable name and coordinates, with no inference required at all?

**Method:** Direct join on `stop_id`. This is the "free win": no algorithm, just enrichment.

In [2]:
import pandas as pd
import duckdb
df_deduped = pd.read_parquet('telemetry_sample_N1.parquet')

GTFS_STATIC_PATH = '../gtfs_static/MBTA_GTFS'

query_primary = f"""
    SELECT
        p.vehicle_id,
        p.trip_id,
        p.route_id,
        p.timestamp,
        p.current_status,
        p.current_stop_sequence,
        p.stop_id,
        s.stop_name,
        s.stop_lat,
        s.stop_lon
    FROM df_deduped AS p
    LEFT JOIN read_csv_auto('{GTFS_STATIC_PATH}/stops.txt', types={{'stop_id': 'VARCHAR'}}) AS s
        ON p.stop_id = s.stop_id
    ORDER BY p.vehicle_id, p.timestamp
"""

df_primary = duckdb.sql(query_primary).df()

has_stop = df_primary['stop_id'].notna().sum()
total = len(df_primary)
print(f"Pings with a directly-reported stop_id: {has_stop} / {total} ({has_stop/total*100:.1f}%)")
df_primary.head(10)


Pings with a directly-reported stop_id: 18530 / 18629 (99.5%)


,vehicle_id,trip_id,route_id,timestamp,current_status,current_stop_sequence,stop_id,stop_name,stop_lat,stop_lon
0,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:19:15-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
1,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:19:51-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
2,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:20:17-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
3,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:20:24-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
4,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:20:48-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
5,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:20:52-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
6,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:21:26-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
7,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:21:50-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
8,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:22:21-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229
9,1700,NorthBase-830295-77,CR-Newburyport,2026-08-18 17:22:34-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.04229


**Result:** An overwhelming 99.5% of pings (18,530 out of 18,629) contain a directly reported `stop_id` provided by the MBTA feed natively.

**Engineering Decision:** With 99.5% native coverage, building a custom spatial-distance (lat/lon) inference algorithm as our default strategy is completely unnecessary and would introduce unjustified computational overhead. I will trust the API's payload, delegating stop-resolution to the MBTA's server-side logic.

## B. Validating the Primary Strategy

**Question:** Is MBTA's own reported `current_stop_sequence` internally consistent? Does it increase monotonically as a vehicle progresses along a trip, or does it ever jump backwards, which would mean the raw field can't be trusted blindly either?

**Method:** For each `(vehicle_id, trip_id)`, sort by timestamp and check whether `current_stop_sequence` is non-decreasing. A negative diff means the reported sequence went backwards, and it would be worth knowing before treating this field as ground truth everywhere.

In [3]:
df_seq = df_primary.dropna(subset=['current_stop_sequence']).copy()
df_seq = df_seq.sort_values(['vehicle_id', 'trip_id', 'timestamp'])

df_seq['seq_diff'] = df_seq.groupby(['vehicle_id', 'trip_id'])['current_stop_sequence'].diff()

backwards = df_seq[df_seq['seq_diff'] < 0]
print(f"Total consecutive same-trip observations: {len(df_seq)}")
print(f"Backwards stop_sequence jumps: {len(backwards)} ({len(backwards)/len(df_seq)*100:.2f}%)")

if len(backwards) > 0:
    print("\nSample of backwards jumps:")
    display(backwards[['vehicle_id', 'trip_id', 'timestamp', 'current_stop_sequence', 'seq_diff']].head(10))


Total consecutive same-trip observations: 18530
Backwards stop_sequence jumps: 6 (0.03%)

Sample of backwards jumps:


,vehicle_id,trip_id,timestamp,current_stop_sequence,seq_diff
2389,R-548B3D99,76734489,2026-08-18 17:21:10-06:00,50.0,-10.0
2586,R-548B4348,76734463,2026-08-18 17:27:35-06:00,210.0,-10.0
5268,y1318,77141226,2026-08-18 17:24:25-06:00,5.0,-1.0
5352,y1331,77141135,2026-08-18 17:23:26-06:00,6.0,-1.0
12196,y1987,76791686,2026-08-18 17:24:54-06:00,3.0,-1.0
18532,y4229,76792742,2026-08-18 17:23:25-06:00,1.0,-1.0


**Result:**  Out of 18,530 consecutive same-trip observations, 6 instances (0.03%) exhibited backwards sequence jumps (`seq_diff < 0`).

**Engineering Decision:**  The reported `current_stop_sequence` is remarkably consistent (>99.9% monotonic). The few non-monotonic anomalies (0.03%) represent edge cases, rather than systematic feed corruption. It can be safely trusted `current_stop_sequence` directly in the Python analytics engine without needing custom monotonicity correction heuristics or sequence smoothing pipelines.

## C. Scoping the Fallback — Can Distance Inference Even Help?

**Question:** `01_data_quality_and_frequency.ipynb` (Section F) found that missing `stop_id` concentrates almost entirely in `Shuttle-Generic*` routes with `BL`-prefixed trip IDs, likely because these are replacement-service trips with no corresponding entry in `stop_times.txt` at all. If that's true, a distance-based fallback matcher can't help these rows either, since it still needs `stop_times.txt` to know which stops belong to a given trip.

**Method:** For every trip_id among the null-stop_id pings, check whether it exists in `stop_times.txt` at all. This directly settles whether building a fallback matcher is worth doing, before writing one.

In [4]:
null_stop_trip_ids = (
    df_deduped[df_deduped['stop_id'].isna()]['trip_id']
    .dropna()
    .unique()
    .tolist()
)

query_fallback_feasibility = f"""
    WITH null_stop_trips AS (
        SELECT UNNEST($trip_ids) AS trip_id
    )
    SELECT
        COUNT(*) AS total_null_stop_trips,
        COUNT(*) FILTER (
            WHERE trip_id IN (
                SELECT DISTINCT trip_id
                FROM read_csv_auto('{GTFS_STATIC_PATH}/stop_times.txt', types={{'trip_id': 'VARCHAR'}})
            )
        ) AS resolvable_in_static_schedule
    FROM null_stop_trips
"""

result = duckdb.sql(query_fallback_feasibility, params={'trip_ids': null_stop_trip_ids}).df()
result['pct_resolvable'] = result['resolvable_in_static_schedule'] / result['total_null_stop_trips'] * 100
result


,total_null_stop_trips,resolvable_in_static_schedule,pct_resolvable
0,21,2,9.52381


**Result:** 9.5% (2 out of 21) of the trip IDs missing a `stop_id` actually exist in `stop_times.txt`. Over 90% (19 trips) have no static schedule entries at all.

**Engineering Decision:**  A distance-based fallback matcher is rejected for the MVP. Spatial inference cannot compute schedule deviations for trips that do not exist in `stop_times.txt` in the first place. The missing stop data is concentrated in ad-hoc replacement shuttles (`Shuttle-Generic*`). 

Excluding unscheduled/shuttle trips from stop-level schedule deviation metrics is a justified scope boundary, preventing dead-code complexity and ensuring the analytics engine focuses strictly on revenue-service trips with measurable schedules.

### I. Summary of Engineering Decisions

*Filled once every section above has been run against fresh data.*

| Decision | Value | Source |
| --- | --- | --- |
| **Primary Strategy Coverage** | **99.5%** natively reported `stop_id` via MBTA API payload. | Section A |
| **Sequence Monotonicity** | **>99.9% consistent**. Backwards jumps (0.03%) are rare edge cases; no smoothing heuristics needed. | Section B |
| **Fallback Feasibility** | **Rejected.** 90.5% of trips with missing stop data do not even exist in the static schedule. | Section C |
| **Final Architecture** | Direct in-memory `LEFT JOIN` on `stop_id`. **Zero spatial/distance inference required.** Unscheduled shuttles scoped out. | Sections A & C |